# Sprint 4 — Hyperparameter Tuning — Recall First

Este notebook optimiza los mejores modelos baseline identificados previamente:

- DecisionTree
- LinearSVM
- LogisticRegression

La selección prioriza:

1. **Recall**, para detectar la mayor cantidad posible de `IsBadBuy = 1`.
2. **F2**, como métrica secundaria con mayor peso en recall.
3. **Precision**, como control para no marcar demasiados autos buenos como malos.

Además del tuning de hiperparámetros, se aplica **threshold tuning** con predicciones out-of-fold para no depender del umbral default `0.50`.

Los modelos avanzados de boosting, como **XGBoost** y **LightGBM**, se evalúan en el siguiente notebook.

In [17]:
from pathlib import Path
import sys
import json
import warnings

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

from src.config import *
from src.io_utils import load_kick_data
from src.preprocessing import prepare_features, split_X_y
from src.models import get_cv
from src.tuning import randomized_tune
from src.evaluation import threshold_tuning_cv, select_best_threshold, get_oof_scores
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

N_ITER = 15
N_JOBS = 1
CANDIDATE_MODELS = [
    "DecisionTree",
    "LinearSVM",
    "LogisticRegression",
]
cv = get_cv()

In [18]:
# Cargar muestra si existe; si no, cargar raw y muestrear estratificado.
sample_path = PROCESSED_DIR / "train_sample.csv"

if sample_path.exists():
    df_model = pd.read_csv(sample_path)
else:
    df_model = load_kick_data()
    df_model = df_model.dropna(subset=[TARGET]).copy()

df_model = prepare_features(df_model)
X, y = split_X_y(df_model)

if len(X) > SAMPLE_SIZE:
    X_sample, _, y_sample, _ = train_test_split(
        X,
        y,
        train_size=SAMPLE_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )
else:
    X_sample, y_sample = X.copy(), y.copy()

print("X_sample:", X_sample.shape)
display(y_sample.value_counts().rename("count").to_frame())
display(y_sample.value_counts(normalize=True).rename("share").to_frame())

X_sample: (20000, 40)


,count
IsBadBuy,
0,17540
1,2460


,share
IsBadBuy,
0,0.877
1,0.123


In [19]:
# Tunear por recall -> F2 -> precision usando src.tuning.recall_refit dentro de randomized_tune().
tuning_rows = []
searches = {}

for model_name in CANDIDATE_MODELS:
    row, search = randomized_tune(
        model_name,
        X_sample,
        y_sample,
        n_iter=N_ITER,
        n_jobs=N_JOBS,
        cv=cv,
    )
    tuning_rows.append(row)
    searches[model_name] = search

tuning_results = (
    pd.DataFrame(tuning_rows)
    .sort_values(["best_recall", "best_f2", "best_precision"], ascending=[False, False, False])
    .reset_index(drop=True)
)

tuning_path = REPORTS_DIR / "tuning_results_recall_first.csv"
tuning_results.to_csv(tuning_path, index=False)

display(tuning_results[[
    "model", "best_recall", "best_f2", "best_precision",
    "train_recall", "recall_gap_train_minus_cv", "seconds", "path"
]])
print("Guardado:", tuning_path)

Fitting 5 folds for each of 15 candidates, totalling 75 fits
Fitting 5 folds for each of 15 candidates, totalling 75 fits
Fitting 5 folds for each of 15 candidates, totalling 75 fits


,model,best_recall,best_f2,best_precision,train_recall,recall_gap_train_minus_cv,seconds,path
0,DecisionTree,0.613008,0.456183,0.228440,0.697154,0.084146,53.1,/Users/alexandralozano/dp261-g1/models/tuned_DecisionTree.pkl
1,LogisticRegression,0.609756,0.482136,0.262460,0.656606,0.046850,37.0,/Users/alexandralozano/dp261-g1/models/tuned_LogisticRegression.pkl
2,LinearSVM,0.603252,0.481969,0.267182,0.647154,0.043902,47.3,/Users/alexandralozano/dp261-g1/models/tuned_LinearSVM.pkl


Guardado: /Users/alexandralozano/dp261-g1/reports/tuning_results_recall_first.csv


In [ ]:
# Buscar threshold con predicciones out-of-fold; no usar threshold default 0.50.
threshold_frames = []

for model_name, search in searches.items():
    results = threshold_tuning_cv(
        model_name,
        search.best_estimator_,
        X_sample,
        y_sample,
        cv=cv,
        n_jobs=N_JOBS,
    )
    threshold_frames.append(results)

threshold_results = pd.concat(threshold_frames, ignore_index=True)

threshold_results_path = REPORTS_DIR / "threshold_tuning_all_results_recall_first.csv"
threshold_results.to_csv(threshold_results_path, index=False)

best_thresholds = (
    threshold_results
    .groupby("model", group_keys=False)
    .apply(lambda g: select_best_threshold(g))
    .reset_index(drop=True)
    .sort_values(["recall", "f2", "precision"], ascending=[False, False, False])
    .reset_index(drop=True)
)

best_thresholds_path = REPORTS_DIR / "threshold_tuning_best_recall_first.csv"
best_thresholds.to_csv(best_thresholds_path, index=False)

display(best_thresholds[[
    "model", "threshold", "recall", "f2", "precision",
    "positive_rate", "tp", "fp", "fn", "tn"
]])
print("Guardado:", threshold_results_path)
print("Guardado:", best_thresholds_path)

,model,threshold,recall,f2,precision,positive_rate,tp,fp,fn,tn
0,LogisticRegression,0.02,1.00000,0.412489,0.123129,0.99895,2460,17519,0,21
1,LinearSVM,0.69,1.00000,0.412323,0.123055,0.99955,2460,17531,0,9
2,DecisionTree,0.01,0.94878,0.420192,0.130151,0.89665,2334,15599,126,1941


Guardado: /Users/alexandralozano/dp261-g1/reports/threshold_tuning_all_results_recall_first.csv
Guardado: /Users/alexandralozano/dp261-g1/reports/threshold_tuning_best_recall_first.csv


In [22]:
# Ranking final: hiperparámetros + mejor threshold.
final_ranking = tuning_results.merge(
    best_thresholds.add_prefix("threshold_"),
    left_on="model",
    right_on="threshold_model",
    how="left",
)

final_ranking = (
    final_ranking
    .sort_values(["threshold_recall", "threshold_f2", "threshold_precision"], ascending=[False, False, False])
    .reset_index(drop=True)
)

final_ranking_path = REPORTS_DIR / "final_model_ranking_recall_first_with_threshold.csv"
final_ranking.to_csv(final_ranking_path, index=False)

display(final_ranking[[
    "model", "best_recall", "best_f2", "best_precision",
    "threshold_threshold", "threshold_recall", "threshold_f2", "threshold_precision",
    "threshold_positive_rate", "threshold_tp", "threshold_fp", "threshold_fn", "threshold_tn",
    "path"
]])
print("Guardado:", final_ranking_path)

,model,best_recall,best_f2,best_precision,threshold_threshold,threshold_recall,threshold_f2,threshold_precision,threshold_positive_rate,threshold_tp,threshold_fp,threshold_fn,threshold_tn,path
0,LogisticRegression,0.609756,0.482136,0.262460,0.02,1.00000,0.412489,0.123129,0.99895,2460,17519,0,21,/Users/alexandralozano/dp261-g1/models/tuned_LogisticRegression.pkl
1,LinearSVM,0.603252,0.481969,0.267182,0.69,1.00000,0.412323,0.123055,0.99955,2460,17531,0,9,/Users/alexandralozano/dp261-g1/models/tuned_LinearSVM.pkl
2,DecisionTree,0.613008,0.456183,0.228440,0.01,0.94878,0.420192,0.130151,0.89665,2334,15599,126,1941,/Users/alexandralozano/dp261-g1/models/tuned_DecisionTree.pkl


Guardado: /Users/alexandralozano/dp261-g1/reports/final_model_ranking_recall_first_with_threshold.csv


In [23]:
# Revisar matriz de confusión del mejor modelo final.
best_final = final_ranking.iloc[0]
best_model_name = best_final["model"]
best_threshold = float(best_final["threshold_threshold"])
best_estimator = searches[best_model_name].best_estimator_

scores, score_type = get_oof_scores(
    best_estimator,
    X_sample,
    y_sample,
    cv=cv,
    n_jobs=N_JOBS,
)

y_pred = (scores >= best_threshold).astype(int)
cm = confusion_matrix(y_sample, y_pred, labels=[0, 1])

print("Mejor modelo:", best_model_name)
print("Threshold:", best_threshold)
print("Score type:", score_type)
display(pd.DataFrame(cm, index=["real_0", "real_1"], columns=["pred_0", "pred_1"]))
print(classification_report(y_sample, y_pred, zero_division=0))

Mejor modelo: LogisticRegression
Threshold: 0.02
Score type: predict_proba


,pred_0,pred_1
real_0,21,17519
real_1,0,2460


              precision    recall  f1-score   support

           0       1.00      0.00      0.00     17540
           1       0.12      1.00      0.22      2460

    accuracy                           0.12     20000
   macro avg       0.56      0.50      0.11     20000
weighted avg       0.89      0.12      0.03     20000



In [24]:
# Guardar metadata del modelo final.
final_metadata = {
    "model_name": best_model_name,
    "threshold": best_threshold,
    "score_type": score_type,
    "priority": "recall_then_f2_then_precision",
    "model_path": best_final["path"],
    "ranking_path": str(final_ranking_path),
}

metadata_path = MODELS_DIR / "final_recall_first_model_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(final_metadata, f, indent=2, ensure_ascii=False)

print("Metadata guardada:", metadata_path)
final_metadata

Metadata guardada: /Users/alexandralozano/dp261-g1/models/final_recall_first_model_metadata.json


{'model_name': 'LogisticRegression',
 'threshold': 0.02,
 'score_type': 'predict_proba',
 'priority': 'recall_then_f2_then_precision',
 'model_path': '/Users/alexandralozano/dp261-g1/models/tuned_LogisticRegression.pkl',
 'ranking_path': '/Users/alexandralozano/dp261-g1/reports/final_model_ranking_recall_first_with_threshold.csv'}